# ✂️ Lesson 06: Chunking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/06_chunking.ipynb)

> **How you cut the text decides what the search can find. A bad cut hides a good answer.**

---

## 🎯 Goal
By the end of this lesson you can split documents into chunks with several strategies, keep the metadata on every chunk, and explain when to use which.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~25 minutes |
| 📚 **Prerequisites** | [Lesson 05](05_metadata.ipynb) |
| 🔑 **API key** | Not needed |
| 🛠 **Libraries** | `pdfplumber`, `langchain-core`, `langchain-text-splitters` |

In [ ]:
!pip install -q -U pdfplumber langchain-core langchain-text-splitters

In [1]:
from pathlib import Path
import urllib.request
import re
import unicodedata
from collections import Counter
import pdfplumber
from langchain_core.documents import Document

PDF_URL = "https://raw.githubusercontent.com/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/main/data/pdf1.pdf"

pdf_path = next((p for p in [Path("data/pdf1.pdf"), Path("../data/pdf1.pdf")] if p.exists()), None)
if pdf_path is None:
    pdf_path = Path("data/pdf1.pdf")
    pdf_path.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(PDF_URL, pdf_path)

# Lesson 03: parse into sections
lines = []
with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages):
        for line in page.extract_text_lines():
            size = round(max(char["size"] for char in line["chars"]), 1)
            lines.append({"text": line["text"], "size": size, "page": page_number})

body_size = Counter(line["size"] for line in lines).most_common(1)[0][0]

sections = []
current = None
for line in lines:
    if line["size"] > body_size:
        current = {"title": line["text"], "page": line["page"], "text": []}
        sections.append(current)
    elif current is not None:
        current["text"].append(line["text"])
sections = [section for section in sections if section["text"]]

# Lesson 04: clean
def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("(cid:127)", "\n•")
    text = re.sub(r"Full[n■]time", "Full-time", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

cleaned_documents = [
    Document(
        page_content=clean_text(" ".join(section["text"])),
        metadata={"source": pdf_path.name, "page": section["page"], "section": section["title"]},
    )
    for section in sections
]

print("documents:", len(cleaned_documents))
print("lengths:", [len(doc.page_content) for doc in cleaned_documents])

documents: 4
lengths: [427, 360, 558, 462]


## 🤔 Why split at all?

In Lesson 01 we saw that sending a whole document is expensive and noisy. In RAG we want to retrieve **only the part that answers the question**. So we split documents into **chunks**.

Two settings matter in every strategy:

| Setting | Meaning | Too small | Too large |
|---|---|---|---|
| `chunk_size` | max length of a chunk | loses context ("20 days" without saying *what*) | brings noise, costs more |
| `chunk_overlap` | how much neighbouring chunks share | an answer cut at the border is lost | duplicates text, grows the index |

There is **no universal best size**. You choose it by measuring (we do that in the Evaluation lessons).

Our sections are only 360 to 558 characters long, so we use a deliberately **small `chunk_size=200`** to make the differences visible. Real documents use bigger values.

In [2]:
from langchain_text_splitters import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(separator="", chunk_size=200, chunk_overlap=40)
fixed_chunks = fixed_splitter.split_documents(cleaned_documents)

print("chunks:", len(fixed_chunks))
print("lengths:", [len(chunk.page_content) for chunk in fixed_chunks])
print()
for chunk in fixed_chunks[4:7]:
    print(chunk.metadata["section"])
    print("  ", repr(chunk.page_content[:80]))

chunks: 12
lengths: [200, 199, 107, 200, 200, 199, 200, 200, 78, 200, 200, 142]

Workplace Conduct Policy
   'nizational values, collaborate effectively, and communicate constructively. Disr'
Paid Time Off (PTO) Policy
   'Full-time employees accrue PTO according to the following schedule:\n• 0–1 years '
Paid Time Off (PTO) Policy
   'ays per year (1.25 days per month)\n• 3+ years of service: 20 days per year (1.67'


## 🔪 Strategy 2: Recursive

Fixed-size cut words in half. The **recursive** splitter is smarter: it has a list of separators, from "big" to "small":

`paragraph break` ➜ `new line` ➜ `space` ➜ `any character`

It first tries to cut at the biggest separator that keeps the chunk under `chunk_size`. Only if that is impossible does it move to a smaller one. So chunks tend to end at natural boundaries.

We use the same `chunk_size=200` and `chunk_overlap=40`, so the comparison is fair.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
recursive_chunks = recursive_splitter.split_documents(cleaned_documents)

print("chunks:", len(recursive_chunks))
print("lengths:", [len(chunk.page_content) for chunk in recursive_chunks])
print()
for chunk in recursive_chunks[4:7]:
    print(chunk.metadata["section"])
    print("  ", repr(chunk.page_content[:80]))

chunks: 13
lengths: [196, 194, 115, 190, 196, 45, 194, 199, 193, 39, 196, 198, 137]

Workplace Conduct Policy
   'organizational values, collaborate effectively, and communicate constructively. '
Workplace Conduct Policy
   'Violations may result in disciplinary action.'
Paid Time Off (PTO) Policy
   'Full-time employees accrue PTO according to the following schedule:\n• 0–1 years '


### 👀 Compare

| | Fixed-size | Recursive |
|---|---|---|
| Cuts inside words | yes | no (only as a last resort) |
| Respects lines and sentences | no | yes, as much as possible |
| Chunk lengths | almost always exactly `chunk_size` | vary, but stay under it |
| Speed and simplicity | simplest | still simple |

Look at the chunks above: the recursive ones start with whole words. A chunk that starts with `nizational` is much harder to match with a question about "organizational values".

In [4]:
pto_fixed = [chunk.page_content for chunk in fixed_chunks if "PTO" in chunk.metadata["section"]]
pto_recursive = [chunk.page_content for chunk in recursive_chunks if "PTO" in chunk.metadata["section"]]

print("FIXED:")
for text in pto_fixed:
    print("  |", repr(text[:60]), "...", repr(text[-30:]))

print()
print("RECURSIVE:")
for text in pto_recursive:
    print("  |", repr(text[:60]), "...", repr(text[-30:]))

FIXED:
  | 'Full-time employees accrue PTO according to the following sc' ... 'ear (1.25 days per month)\n• 3+'
  | 'ays per year (1.25 days per month)\n• 3+ years of service: 20' ... ' be submitted in advance throu'
  | 'sts should be submitted in advance through the HR system unl' ... 'nces exceeding 5 consecutive b'
  | 'ended absences exceeding 5 consecutive business days require' ... 'days require manager approval.'

RECURSIVE:
  | 'Full-time employees accrue PTO according to the following sc' ... 'per year (1.25 days per month)'
  | '• 3+ years of service: 20 days per year (1.67 days per month' ... 'he HR system unless related to'
  | 'through the HR system unless related to an emergency. Employ' ... ' business days require manager'
  | 'business days require manager approval.' ... 'days require manager approval.'


## 🧩 Strategy 3: Document-based (use the structure)

In Lesson 03 we worked hard to find **sections**. Why throw that away? The idea: **each section is already a natural chunk.** We only split it further if it is too long.

Benefits: a policy stays in one piece (the number "20 days" stays next to what it is about), and the section title is already in the metadata.

Risk: it depends on Lesson 03 having found the headings correctly, and a very long section needs a fallback.

In [5]:
MAX_CHARS = 600

document_chunks = []
for doc in cleaned_documents:
    if len(doc.page_content) <= MAX_CHARS:
        document_chunks.append(doc)
    else:
        document_chunks.extend(recursive_splitter.split_documents([doc]))

print("chunks:", len(document_chunks))
for chunk in document_chunks:
    print(len(chunk.page_content), "|", chunk.metadata["section"])

chunks: 4
427 | Non-Disclosure Agreement (NDA) Policy
360 | Workplace Conduct Policy
558 | Paid Time Off (PTO) Policy
462 | Travel & Expense Policy


## 🔢 Strategy 4: Token-based

Language models don't read characters. They read **tokens** (pieces of words). A model's limits and prices are measured in tokens, so sometimes we want `chunk_size` in tokens too.

Rule of thumb: for English, a token is a few characters on average, but this varies by language and by tokenizer.

⚠️ A token splitter can cut **anywhere inside a word**, because tokens don't always match word boundaries. And the tokenizer must match your model. The one below (`cl100k_base`) belongs to some OpenAI models, not to our local embedding model from Lesson 00.

In [ ]:
!pip install -q -U tiktoken

In [6]:
from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(encoding_name="cl100k_base", chunk_size=50, chunk_overlap=10)
token_chunks = token_splitter.split_documents(cleaned_documents)

print("chunks:", len(token_chunks))
print("character lengths:", [len(chunk.page_content) for chunk in token_chunks])
print("first chunk:", repr(token_chunks[0].page_content))

chunks: 10
character lengths: [301, 185, 314, 97, 174, 168, 246, 101, 296, 216]
first chunk: 'Employees must protect confidential information belonging to the company, its clients, and partners. This includes, but is not limited to, product roadmaps, customer data, internal communications, proprietary algorithms, financial information, and unreleased features. Confidential information may not'


In [7]:
strategies = {
    "fixed": fixed_chunks,
    "recursive": recursive_chunks,
    "document": document_chunks,
    "token": token_chunks,
}

for name, chunks in strategies.items():
    lengths = [len(chunk.page_content) for chunk in chunks]
    print(f"{name:10} chunks={len(chunks):3}  avg={sum(lengths) / len(lengths):6.1f}  min={min(lengths):4}  max={max(lengths):4}")

fixed      chunks= 12  avg= 177.1  min=  78  max= 200
recursive  chunks= 13  avg= 160.9  min=  39  max= 199
document   chunks=  4  avg= 451.8  min= 360  max= 558
token      chunks= 10  avg= 209.8  min=  97  max= 314


### 👀 What the numbers say

| Strategy | Chunks | Idea | Weak point we saw |
|---|---|---|---|
| **Fixed-size** | 12 | cut every N characters | cuts inside words (`nizational`, `ays per year`) and splits a bullet in two |
| **Recursive** | 13 | cut at the biggest natural boundary | produced a tiny 45-character chunk |
| **Document-based** | 4 | one chunk per section | only good because our sections are short |
| **Token-based** | 10 | cut every N tokens | can stop in the middle of a sentence |

> ⚠️ These numbers describe *our tiny document with `chunk_size=200`*. They say how the strategies **behave**, not which one is **best**. "Best" depends on your documents and your questions, and we measure it in the Evaluation lessons.

## ⚠️ Things to watch out for

- **There is no universal chunk size.** The values here (200 characters, 50 tokens, `MAX_CHARS = 600`) were chosen to make differences visible, not because they are good defaults.
- **Metadata survives splitting**, because `split_documents` copies it to every chunk. But a chunk *loses its context*: a chunk saying "20 days per year" doesn't say it is about PTO unless the section title travels with it. That's why we keep `section` in the metadata.
- **Semantic chunking** (cutting where the *meaning* changes) is not covered here: it needs embeddings, which we meet in the next lesson. We'll come back to it.

## 📌 Summary

- **Chunking** splits documents into pieces small enough to retrieve precisely and big enough to keep their meaning.
- Two settings in every strategy: `chunk_size` and `chunk_overlap`. There is no universal best value: measure.
- **Fixed-size** is the simplest but cuts inside words. **Recursive** cuts at natural boundaries. **Document-based** reuses the structure we found in Lesson 03. **Token-based** counts the way a model counts.
- `split_documents` keeps the **metadata** on every chunk.
- We compared strategies by looking at real chunks and numbers (count, average, min, max), not by guessing.

## ✍️ Exercise (10 minutes)

Run the recursive splitter with `chunk_size=100` and then `chunk_size=400` (keep `chunk_overlap=40`).

1. Print the number of chunks and the minimum length for each setting.
2. Which setting produces the smallest chunk? Why do you think so?

**Success:** you can explain in one sentence what changes when `chunk_size` gets smaller or larger.

💡 Solution: `solutions/06_chunking.ipynb` (try on your own first)

## 🔥 Challenge (optional)

Imagine the question: *"How many PTO days do I get after 3 years?"*

1. In `fixed_chunks`, find the chunk(s) that contain the text `20 days`. Does that chunk also say it is about PTO and "3+ years"?
2. Do the same for `recursive_chunks` and `document_chunks`.
3. Which strategy would give a model the most useful chunk for this question?

Keep your answer: in Lesson 17 we measure this properly instead of guessing.

## 📖 Next

We now have clean, labeled chunks. To search them by *meaning*, we must turn each chunk into numbers. That is an **embedding**.

**➡️ Next lesson:** [07 Embeddings](07_embeddings.ipynb)